In [9]:
import scipy.io as sio
import numpy as np
import os
import glob
from matplotlib import pyplot as plt
import re
import random
import json
import pandas as pd

In [10]:
def save_data(save_path, filename, tf):
    if os.path.exists(os.path.join(save_path, filename)):
        return
    else:
        sio.savemat(os.path.join(save_path, filename), tf)

In [11]:
data_path = "/path/to/mae/data/fixed_data_final"
target_files = glob.glob(os.path.join(data_path, "*.mat"))
lco_files = [i for i in target_files if "LCO" in i]
nmc_files = [i for i in target_files if "NMC" in i]
print(len(lco_files))
print(len(nmc_files))

27
27


In [ ]:
# mae_train_save_path = "/path/to/mae/data/diff_num_train_data/mae_train"

# #---生成mae_train数据---#
# diff_num = 5 # 5, 10, 15, 19
# mae_train = "{mae_train_save_path}/mix_{diff_num}".format(mae_train_save_path=mae_train_save_path, diff_num=diff_num)
# os.makedirs(mae_train, exist_ok=True)

# ----- 分组 ---- #
file_groups = {}

for file_path in lco_files + nmc_files:
        filename = os.path.basename(file_path)
        
        # Extract battery name and state
        # Pattern: {battery_name}_{state}_wo_{number}.mat
        parts = filename.split('_wo_')[0]
        battery_name, state = parts.split('_', 1)
        
        # Create key for grouping
        group_key = (battery_name, state)
        
        # Add file to group
        if group_key not in file_groups:
            file_groups[group_key] = []
        file_groups[group_key].append(file_path)

print(file_groups)
# # Generate weights with finer granularity based on expand_factor
# step = 1 / 10  # smaller step size for higher granularity
# weight = []
# for i in range(1, 10):  # more pairs based on expand_factor
#     w1 = round(i * step, 3)  # round to 3 decimal places
#     w2 = round(1 - w1, 3)
#     weight.append((w1, w2))
    


{('LCO#4',
  'discharge'): ['/path/to/mae/data/fixed_data_final/LCO#4_discharge_wo_17.mat', '/path/to/mae/data/fixed_data_final/LCO#4_discharge_wo_14.mat', '/path/to/mae/data/fixed_data_final/LCO#4_discharge_wo_15.mat', '/path/to/mae/data/fixed_data_final/LCO#4_discharge_wo_12.mat', '/path/to/mae/data/fixed_data_final/LCO#4_discharge_wo_10.mat', '/path/to/mae/data/fixed_data_final/LCO#4_discharge_wo_16.mat', '/path/to/mae/data/fixed_data_final/LCO#4_discharge_wo_11.mat'],
 ('LCO#4',
  'charge'): ['/path/to/mae/data/fixed_data_final/LCO#4_charge_wo_6.mat', '/path/to/mae/data/fixed_data_final/LCO#4_charge_wo_2.mat', '/path/to/mae/data/fixed_data_final/LCO#4_charge_wo_8.mat', '/path/to/mae/data/fixed_data_final/LCO#4_charge_wo_3.mat', '/path/to/mae/data/fixed_data_final/LCO#4_charge_wo_0.mat', '/path/to/mae/data/fixed_data_final/LCO#4_charge_wo_4.mat', '/path/to/mae/data/fixed_data_final/LCO#4_charge_wo_7.mat', '/path/to/mae/data/fixed_data_final/LCO#4_charge_wo_5.mat'],
 ('LCO#3',
  'dis

In [12]:
diff_path = "/path/to/mae/data/diff_num_train_data"
regression_data_json_path = os.path.join(diff_path, "regression_data")
os.makedirs(regression_data_json_path, exist_ok=True)


In [13]:
total_data_path = "/path/to/mae/data/mae_train_all"

In [36]:
lco_efficient_nums = ['41', '49', '14', '31', '2', '5', '54', '39', '3', '30', '52', '55', '16', '27', '45', '17', '28', '57', '35', '6', '58', '51', '7', '34', '33']
nmc_efficient_nums = ['31', '4', '5', '37', '30', '52', '41', '46', '12', '15', '24', '47', '25', '22', '50', '59', '51', '33', '8', '16', '18', '42', '21', '28', '17']

In [41]:
data_type = "NMC"

regression_data_save_path = "/path/to/mae/data/diff_num_train_data/regression_data"
diff_num = 15 # 5, 10, 15, 19
regression_train = "{mae_train_save_path}/{data_type}_diff_{diff_num}".format(data_type=data_type, mae_train_save_path=regression_data_save_path, diff_num=diff_num)
os.makedirs(regression_train, exist_ok=True)


nineteen_file_path = "/path/to/mae/data/final_test"
trial_name = os.listdir(os.path.join(nineteen_file_path, data_type))

if data_type == "LCO":
    eff_trials = [trial for trial in trial_name if trial.split('_')[-1] in lco_efficient_nums]
elif data_type == "NMC":
    eff_trials = [trial for trial in trial_name if trial.split('_')[-1] in nmc_efficient_nums]
for trial in eff_trials:
    trial_path = os.path.join(nineteen_file_path, data_type, trial)
    test_data_names = os.listdir(os.path.join(trial_path, "test_data"))
    final_test_path = [os.path.join(total_data_path, name) for name in test_data_names if name.endswith('.mat')]
    if data_type == "LCO":
        train_data_names = [os.path.basename(f) for f in lco_files if os.path.basename(f) not in test_data_names]
    elif data_type == "NMC":
        train_data_names = [os.path.basename(f) for f in nmc_files if os.path.basename(f) not in test_data_names]
    selected_files = random.sample(train_data_names, diff_num)
    final_train_path = []
    file_groups = {}
    for file_path in selected_files:
        filename = os.path.basename(file_path)
        final_train_path.append(os.path.join(total_data_path, filename))
        # Extract battery name and state
        # Pattern: {battery_name}_{state}_wo_{number}.mat
        parts = filename.split('_wo_')[0]
        battery_name, state = parts.split('_', 1)
        
        # Create key for grouping
        group_key = (battery_name, state)
        
        # Add file to group
        if group_key not in file_groups:
            file_groups[group_key] = []
        file_groups[group_key].append(file_path)
    
    step = 1 / 10  # smaller step size for higher granularity
    weight = []
    for i in range(1, 10):  # more pairs based on expand_factor
        w1 = round(i * step, 3)  # round to 3 decimal places
        w2 = round(1 - w1, 3)
        weight.append((w1, w2))
        
    
    for (battery, state), files in file_groups.items():
        files = sorted(files)
        if len(files) < 2:
            print(f"Not enough files for {battery}_{state} to perform mixup.")
            continue
        for idx in range(len(files)-1):
            
            stage1 = int(re.search(r'wo_(\d+)', files[idx]).group(1))
            stage2 = int(re.search(r'wo_(\d+)', files[idx+1]).group(1))
            for wt1, wt2 in weight:
                new_filename = f"{battery}_{state}_wo_{stage1}_w{wt1}_{stage2}_w{wt2}.mat"
                final_train_path.append(os.path.join(total_data_path, new_filename))
    trial_json = {
        data_type: {
            "train": final_train_path,
            "test": final_test_path
        }
    }
    
    
    with open(os.path.join(regression_train, f'{trial}.json'), 'w') as f:
        json.dump(trial_json, f, indent=4)
    
    
    


Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#2_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_discharge to perform mixup.
Not enough files for NMC#4_discharge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_discharge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_discharge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.
Not enough files for NMC#4_discharge to perform mixup.
Not enough files for NMC#4_charge to perform mixup.


In [15]:
lco_efficient_nums = [str(i) for i in range(32)]
ex_list = ["0", "10", "17", "19", "21", "26", "28"]
lco_efficient_nums = [num for num in lco_efficient_nums if num not in ex_list]
len(lco_efficient_nums)

25

In [34]:
data_type = "LCO"

regression_data_save_path = "/path/to/mae/data/diff_num_train_data/regression_data"
diff_num = 5 # 5, 10, 15, 19
regression_train = "{mae_train_save_path}/{data_type}_diff_{diff_num}".format(data_type=data_type, mae_train_save_path=regression_data_save_path, diff_num=diff_num)
os.makedirs(regression_train, exist_ok=True)


nineteen_file_path = "/path/to/mae/data/LCO_test_vs_ablation_mixuptest"
eff_trials = os.listdir(nineteen_file_path)
for trial in eff_trials:
    trial_json = os.path.join(nineteen_file_path, trial)
    json_data = json.load(open(trial_json, 'r'))
    test_data_names = json_data[data_type]['test']
    test_data_names = [os.path.basename(name) for name in test_data_names]
    final_test_path = [os.path.join(total_data_path, name) for name in test_data_names if name.endswith('.mat')]
    if data_type == "LCO":
        train_data_names = [os.path.basename(f) for f in lco_files if os.path.basename(f) not in test_data_names]
    elif data_type == "NMC":
        train_data_names = [os.path.basename(f) for f in nmc_files if os.path.basename(f) not in test_data_names]

    intersection = set(test_data_names).intersection(set(train_data_names))
    if intersection:
        print("Intersection found between test_data_names and train_data_names:", intersection)
    else:
        # print("No intersection found between test_data_names and train_data_names.")
        pass
    selected_files = random.sample(train_data_names, diff_num)
    final_train_path = []
    file_groups = {}
    for file_path in selected_files:
        filename = os.path.basename(file_path)
        final_train_path.append(os.path.join(total_data_path, filename))
        # Extract battery name and state
        # Pattern: {battery_name}_{state}_wo_{number}.mat
        parts = filename.split('_wo_')[0]
        battery_name, state = parts.split('_', 1)
        
        # Create key for grouping
        group_key = (battery_name, state)
        
        # Add file to group
        if group_key not in file_groups:
            file_groups[group_key] = []
        file_groups[group_key].append(file_path)
    
    step = 1 / 10  # smaller step size for higher granularity
    weight = []
    for i in range(1, 10):  # more pairs based on expand_factor
        w1 = round(i * step, 3)  # round to 3 decimal places
        w2 = round(1 - w1, 3)
        weight.append((w1, w2))
        
    
    for (battery, state), files in file_groups.items():
        files = sorted(files)
        if len(files) < 2:
            # print(f"Not enough files for {battery}_{state} to perform mixup.")
            continue
        for idx in range(len(files)-1):
            
            stage1 = int(re.search(r'wo_(\d+)', files[idx]).group(1))
            stage2 = int(re.search(r'wo_(\d+)', files[idx+1]).group(1))
            for wt1, wt2 in weight:
                new_filename = f"{battery}_{state}_wo_{stage1}_w{wt1}_{stage2}_w{wt2}.mat"
                final_train_path.append(os.path.join(total_data_path, new_filename))

    trial_json = {
        data_type: {
            "train": final_train_path,
            "test": final_test_path
        }
    }

    
    with open(os.path.join(regression_train, f'{trial}.json'), 'w') as f:
        json.dump(trial_json, f, indent=4)
    
    
    


False

In [2]:
import json
orin_json = "/path/to/mae/data/LCO_test_vs_ablation_mixuptest/trial_1.json"
orin_data = json.load(open(orin_json, 'r'))
compare_json = "/path/to/mae/data/diff_num_train_data/regression_data/LCO_diff_19/trial_1.json.json"
compare_data = json.load(open(compare_json, 'r'))


In [4]:
orin_test = orin_data['LCO']['test']
compare_test = compare_data['LCO']['test']
print(set(orin_test) - set(compare_test))
print(set(compare_test) - set(orin_test))

set()
set()


In [5]:
orin_train = orin_data['LCO']['train']
compare_train = compare_data['LCO']['train']
print(set(orin_train) - set(compare_train))
print(set(compare_train) - set(orin_train))

{'/path/to/mae/data/mae_train_all/LCO#4_discharge_wo_10_w0.4_12_w0.6.mat', '/path/to/mae/data/mae_train_all/LCO#2_charge_wo_2_w0.5_4_w0.5.mat', '/path/to/mae/data/mae_train_all/LCO#4_charge_wo_3_w0.8_4_w0.2.mat', '/path/to/mae/data/mae_train_all/LCO#4_discharge_wo_12_w0.9_15_w0.1.mat', '/path/to/mae/data/mae_train_all/LCO#2_charge_wo_5_w0.8_6_w0.2.mat', '/path/to/mae/data/mae_train_all/LCO#4_charge_wo_5_w0.5_8_w0.5.mat', '/path/to/mae/data/mae_train_all/LCO#4_discharge_wo_10_w0.1_12_w0.9.mat', '/path/to/mae/data/mae_train_all/LCO#2_charge_wo_5_w0.2_6_w0.8.mat', '/path/to/mae/data/mae_train_all/LCO#4_discharge_wo_15_w0.3_16_w0.7.mat', '/path/to/mae/data/mae_train_all/LCO#4_charge_wo_5_w0.9_8_w0.1.mat', '/path/to/mae/data/mae_train_all/LCO#2_charge_wo_2_w0.2_4_w0.8.mat', '/path/to/mae/data/mae_train_all/LCO#4_discharge_wo_12_w0.6_15_w0.4.mat', '/path/to/mae/data/mae_train_all/LCO#2_charge_wo_5_w0.5_6_w0.5.mat', '/path/to/mae/data/mae_train_all/LCO#4_charge_wo_2_w0.6_3_w0.4.mat', '/path/t

In [7]:
len(compare_train)

145

In [6]:
len(orin_train)

154